# <font color='blue'>Tarea 3. </font>
### <font color='red'>17 de diciembre de 2025</font>
### Andrés Padrón Quintana 
### <font color='green'>Módulo 5. DATA SCIENCE AND MACHINE LEARNING APPLIED TO FINANCIAL MARKETS </font>

In [11]:
# Importamos librerías
import csv
import random
from math import exp
import numpy as np
import csv

# Definimos la semilla
SEED = 123
random.seed(SEED)
np.random.seed(SEED)

### Parte 1 (Como lo hicimos en el cuaderno lec03)
Utilizar el dataset de Seeds (se los comparto)
1. Cargar la base de datos.


In [15]:
DATA_PATH = "seeds_dataset.csv"

def load_csv(filename):
    dataset = []
    # utf-8-sig elimina el BOM (\ufeff) si existe
    with open(filename, "r", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        for row in reader:
            if not row:
                continue
            dataset.append(row)
    return dataset

dataset = load_csv(DATA_PATH)


2. Utilizar una función para convertir cada columna a float (excepto la última porque viene categoría:y)


In [16]:
def str_column_to_float(dataset, column):
    for row in dataset:
        val = row[column]
        
        # Si ya es float/int, no intentes strip
        if isinstance(val, (float, int)):
            row[column] = float(val)
        else:
            # Si es string, limpia espacios y BOM
            val = str(val).strip().replace("\ufeff", "")
            row[column] = float(val)

n_cols = len(dataset[0])

# Convertimos todas las columnas excepto la última (y)
for col in range(n_cols - 1):
    str_column_to_float(dataset, col)

print("Ejemplo (features float):", dataset[0])

Ejemplo (features float): [15.26, 14.84, 0.871, 5.763, 3.312, 2.221, 5.22, '1']


3. Utilizar otra función para convertir las clases a valors enteros que inicien en 0. Hay 3 clases en el dataset


In [17]:
def str_column_to_int(dataset, column):
    class_values = [row[column] for row in dataset]
    unique = sorted(set(class_values))          # e.g. ['1','2','3']
    lookup = {value: i for i, value in enumerate(unique)}  # {'1':0,'2':1,'3':2}
    
    for row in dataset:
        row[column] = lookup[row[column]]
    return lookup

# última columna = y
lookup = str_column_to_int(dataset, n_cols - 1)

print("Mapeo de clases:", lookup)
print("Ejemplo (ya con y int):", dataset[0])
print("Clases únicas:", sorted(set(row[-1] for row in dataset)))


Mapeo de clases: {'1': 0, '2': 1, '3': 2}
Ejemplo (ya con y int): [15.26, 14.84, 0.871, 5.763, 3.312, 2.221, 5.22, 0]
Clases únicas: [0, 1, 2]


4. Escalar cada columna con el máximo y mínimo de cada columna (MinMaxScaler)

In [18]:
def dataset_minmax(dataset):
    minmax = []
    n_features = len(dataset[0]) - 1  # sin y
    for i in range(n_features):
        col_values = [row[i] for row in dataset]
        min_val = min(col_values)
        max_val = max(col_values)
        minmax.append((min_val, max_val))
    return minmax

def normalize_dataset(dataset, minmax):
    n_features = len(dataset[0]) - 1
    for row in dataset:
        for i in range(n_features):
            mn, mx = minmax[i]
            denom = (mx - mn)
            row[i] = (row[i] - mn) / denom if denom != 0 else 0.0

minmax = dataset_minmax(dataset)
normalize_dataset(dataset, minmax)

print("Ejemplo (normalizado):", dataset[0])


Ejemplo (normalizado): [0.4409820585457979, 0.5020661157024793, 0.570780399274047, 0.48648648648648646, 0.48610121168923714, 0.18930164220052273, 0.3451501723289019, 0]


5. Definir una nueva función para hacer un 5-folds cross-validation. Calcular el accuracy para cada fold. 

In [19]:
def cross_validation_split(dataset, n_folds):
    dataset_copy = list(dataset)
    random.shuffle(dataset_copy)

    fold_size = int(len(dataset_copy) / n_folds)
    folds = []
    for _ in range(n_folds):
        fold = dataset_copy[:fold_size]
        dataset_copy = dataset_copy[fold_size:]
        folds.append(fold)

    # Repartimos sobrantes si no divide exacto
    i = 0
    while dataset_copy:
        folds[i % n_folds].append(dataset_copy.pop())
        i += 1

    return folds

def accuracy_metric(actual, predicted):
    correct = sum(1 for a, p in zip(actual, predicted) if a == p)
    return 100.0 * correct / len(actual)

def evaluate_algorithm(dataset, algorithm, n_folds, *args):
    folds = cross_validation_split(dataset, n_folds)
    scores = []
    for i, fold in enumerate(folds, start=1):
        train_set = [row for f in folds if f is not fold for row in f]
        test_set  = [row[:] for row in fold]  # copia

        predicted = algorithm(train_set, test_set, *args)
        actual    = [row[-1] for row in fold]

        acc = accuracy_metric(actual, predicted)
        print(f"Fold {i} accuracy = {acc:.2f}%")
        scores.append(acc)

    return scores


6. El modelo será una Red Neuronal (como se vio en este libro con diccionarios) que la entrenen y luego hagan una predicción en el data set (cross-validate)


In [20]:
def initialize_network(n_inputs, n_hidden, n_outputs):
    network = []
    hidden_layer = [{'weights': [random.random() for _ in range(n_inputs + 1)]}
                    for _ in range(n_hidden)]
    network.append(hidden_layer)

    output_layer = [{'weights': [random.random() for _ in range(n_hidden + 1)]}
                    for _ in range(n_outputs)]
    network.append(output_layer)

    return network

def activate(weights, inputs):
    activation = weights[-1]  # bias
    for i in range(len(weights) - 1):
        activation += weights[i] * inputs[i]
    return activation

def transfer(activation):
    return 1.0 / (1.0 + exp(-activation))  # sigmoid

def transfer_derivative(output):
    return output * (1.0 - output)

def forward_propagate(network, row):
    inputs = row[:-1]
    for layer in network:
        new_inputs = []
        for neuron in layer:
            activation = activate(neuron['weights'], inputs)
            neuron['output'] = transfer(activation)
            new_inputs.append(neuron['output'])
        inputs = new_inputs
    return inputs

def backward_propagate_error(network, expected):
    for i in reversed(range(len(network))):
        layer = network[i]
        errors = []

        if i == len(network) - 1:
            # salida
            for j, neuron in enumerate(layer):
                errors.append(neuron['output'] - expected[j])
        else:
            # oculta
            for j in range(len(layer)):
                error = 0.0
                for neuron in network[i + 1]:
                    error += neuron['weights'][j] * neuron['delta']
                errors.append(error)

        for j, neuron in enumerate(layer):
            neuron['delta'] = errors[j] * transfer_derivative(neuron['output'])

def update_weights(network, row, l_rate):
    inputs = row[:-1]
    for i, layer in enumerate(network):
        if i != 0:
            inputs = [neuron['output'] for neuron in network[i - 1]]

        for neuron in layer:
            for j in range(len(inputs)):
                neuron['weights'][j] -= l_rate * neuron['delta'] * inputs[j]
            neuron['weights'][-1] -= l_rate * neuron['delta']  # bias

def train_network(network, train, l_rate, n_epoch, n_outputs):
    for _ in range(n_epoch):
        for row in train:
            outputs = forward_propagate(network, row)
            expected = [0.0 for _ in range(n_outputs)]
            expected[row[-1]] = 1.0
            backward_propagate_error(network, expected)
            update_weights(network, row, l_rate)

def predict(network, row):
    outputs = forward_propagate(network, row)
    return int(np.argmax(outputs))

def back_propagation(train, test, l_rate, n_epoch, n_hidden):
    n_inputs  = len(train[0]) - 1
    n_outputs = len(set(row[-1] for row in train))

    network = initialize_network(n_inputs, n_hidden, n_outputs)
    train_network(network, train, l_rate, n_epoch, n_outputs)

    predictions = []
    for row in test:
        predictions.append(predict(network, row))
    return predictions


7. Los parámetros del modelo serán learning_rate=.01,n_epochs=100, n_hidden=5 (5 neuronas en UNA sólo capa oculta) (los pueden cambiar si mejora el modelo en el cross-validate)


In [21]:
# Definimos los parámetros del modelo dados
n_folds = 5
learning_rate = 0.01
n_epochs = 100
n_hidden = 5

scores = evaluate_algorithm(dataset, back_propagation, n_folds,
                            learning_rate, n_epochs, n_hidden)

print("\nScores:", [round(s, 2) for s in scores])
print(f"Mean Accuracy: {np.mean(scores):.2f}%")
print(f"Std Accuracy:  {np.std(scores):.2f}%")


Fold 1 accuracy = 59.52%
Fold 2 accuracy = 92.86%
Fold 3 accuracy = 61.90%
Fold 4 accuracy = 57.14%
Fold 5 accuracy = 60.98%

Scores: [59.52, 92.86, 61.9, 57.14, 60.98]
Mean Accuracy: 66.48%
Std Accuracy:  13.29%


Existe alta variabilidad entre folds (por ejemplo, un fold con 92.86% vs otros cercanos a 60%). Esto puede ocurrir cuando los folds aleatorios no preservan la proporción de clases en cada partición, generando folds “más fáciles” o “más difíciles”.

### Propuesta
Sin cambiar el modelo (forward/backprop ni actualización de pesos), se puede modificar únicamente la forma en que se construyen los folds para que cada fold mantenga proporciones similares de las 3 clases (stratified cross-validation).

Esto suele reducir la varianza entre folds y dar una estimación de desempeño más estable, especialmente en datasets multicategoría.

In [22]:
def stratified_cross_validation_split(dataset, n_folds):
    # Agrupamos filas por clase
    class_groups = {}
    for row in dataset:
        y = row[-1]
        class_groups.setdefault(y, []).append(row)

    # Mezclamos cada grupo de clase
    for y in class_groups:
        random.shuffle(class_groups[y])

    # Creamos folds vacíos
    folds = [[] for _ in range(n_folds)]

    # Repartimos cada clase de forma round-robin
    for y, rows in class_groups.items():
        for i, row in enumerate(rows):
            folds[i % n_folds].append(row)

    # Mezclamos dentro de cada fold
    for f in folds:
        random.shuffle(f)

    return folds


def evaluate_algorithm_stratified(dataset, algorithm, n_folds, *args):
    folds = stratified_cross_validation_split(dataset, n_folds)
    scores = []
    for i, fold in enumerate(folds, start=1):
        train_set = [row for f in folds if f is not fold for row in f]
        test_set  = [row[:] for row in fold]  # copia

        predicted = algorithm(train_set, test_set, *args)
        actual    = [row[-1] for row in fold]

        acc = accuracy_metric(actual, predicted)
        print(f"Fold {i} accuracy = {acc:.2f}%")
        scores.append(acc)

    return scores


# Re-ejecutamos con los mismos hiperparámetros
scores_strat = evaluate_algorithm_stratified(dataset, back_propagation, 5, 0.01, 100, 5)
print("\nScores (stratified):", [round(s, 2) for s in scores_strat])
print(f"Mean Accuracy (stratified): {np.mean(scores_strat):.2f}%")
print(f"Std Accuracy  (stratified): {np.std(scores_strat):.2f}%")


Fold 1 accuracy = 64.29%
Fold 2 accuracy = 66.67%
Fold 3 accuracy = 66.67%
Fold 4 accuracy = 66.67%
Fold 5 accuracy = 65.85%

Scores (stratified): [64.29, 66.67, 66.67, 66.67, 65.85]
Mean Accuracy (stratified): 66.03%
Std Accuracy  (stratified): 0.93%


No necesariamente subió el promedio, pero sí mejoró drásticamente la estabilidad de la estimación.

Con los hiperparámetros solicitados **(learning_rate = 0.01, n_epochs = 100, n_hidden = 5)** se compararon dos formas de construir los folds en validación cruzada:

**1) 5-Fold CV con partición aleatoria (shuffle + split)**

**Accuracies:** [59.52, 92.86, 61.90, 57.14, 60.98]

**Media:** 66.48%

**Desv. estándar:** 13.29%

Se observa alta variabilidad entre folds, lo cual sugiere que algunos folds pueden quedar con una composición de clases “más favorable” o “menos favorable”, afectando fuertemente el accuracy.

**2) 5-Fold CV estratificado (misma red, mismo entrenamiento; sólo cambia el split)**

**Accuracies:** [64.29, 66.67, 66.67, 66.67, 65.85]

**Media:** 66.03%

**Desv. estándar:** 0.93%

El promedio se mantiene muy similar, pero la variabilidad disminuye drásticamente. Esto indica que el esquema estratificado proporciona una estimación del desempeño más estable al mantener proporciones de clase similares en cada fold.

### Parte 2 (con las librerías de cuadernos pasados): 

Replicar el ejercicio de Keras para corroborar: pueden utilizar wrapper de Scikit-Learn de Keras / directamente keras / scikit-Learn módulo Redes Neuronales

Usamos una arquitectura equivalente: 1 capa oculta con n_hidden=5 neuronas, activación sigmoid y salida softmax para 3 clases.
Evaluamos con 5-fold cross-validation estratificado, reportando accuracy por fold, media y desviación estándar.

In [23]:
import csv
import numpy as np
import random
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler


# Reproducibilidad

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Cargamos CSV -> X, y

DATA_PATH = "seeds_dataset.csv"

def load_seeds_xy(path):
    X_list, y_list = [], []
    with open(path, "r", encoding="utf-8-sig") as f:  # utf-8-sig evita BOM
        reader = csv.reader(f)
        for row in reader:
            if not row:
                continue
            # 7 features
            x = [float(str(v).strip().replace("\ufeff","")) for v in row[:-1]]
            # etiqueta (viene como '1','2','3') -> 0,1,2
            y = int(str(row[-1]).strip()) - 1
            X_list.append(x)
            y_list.append(y)
    return np.array(X_list, dtype=float), np.array(y_list, dtype=int)

X, y = load_seeds_xy(DATA_PATH)

print("X shape:", X.shape, "| y shape:", y.shape)
print("Clases:", np.unique(y))


# Modelo Keras (equivalente a NN)

def build_model(input_dim, n_hidden=5, lr=0.01, n_classes=3):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(n_hidden, activation="sigmoid"),
        tf.keras.layers.Dense(n_classes, activation="softmax")
    ])
    opt = tf.keras.optimizers.SGD(learning_rate=lr)
    model.compile(
        optimizer=opt,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# 5-Fold CV Estratificado

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
accs = []

for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    # MinMaxScaler por fold (fit SOLO en train)
    scaler = MinMaxScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_te_sc = scaler.transform(X_te)

    # Limpiamos el grafo/estado entre folds
    tf.keras.backend.clear_session()
    model = build_model(input_dim=X.shape[1], n_hidden=5, lr=0.01, n_classes=3)

    model.fit(X_tr_sc, y_tr, epochs=100, batch_size=16, verbose=0)
    loss, acc = model.evaluate(X_te_sc, y_te, verbose=0)

    acc_pct = 100.0 * acc
    accs.append(acc_pct)
    print(f"Fold {fold} accuracy (Keras) = {acc_pct:.2f}%")

print("\nScores (Keras):", [round(a, 2) for a in accs])
print(f"Mean Accuracy (Keras): {np.mean(accs):.2f}%")
print(f"Std Accuracy  (Keras): {np.std(accs):.2f}%")


2025-12-12 12:10:20.808731: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


X shape: (209, 7) | y shape: (209,)
Clases: [0 1 2]
Fold 1 accuracy (Keras) = 54.76%
Fold 2 accuracy (Keras) = 61.90%
Fold 3 accuracy (Keras) = 76.19%
Fold 4 accuracy (Keras) = 73.81%
Fold 5 accuracy (Keras) = 78.05%

Scores (Keras): [54.76, 61.9, 76.19, 73.81, 78.05]
Mean Accuracy (Keras): 68.94%
Std Accuracy  (Keras): 9.05%


Con una red neuronal equivalente en Keras **(1 capa oculta con 5 neuronas, activación sigmoid, salida softmax)**, usando 5-fold cross-validation estratificado y escalamiento MinMax por fold:

**Accuracies por fold:** [54.76, 61.90, 76.19, 73.81, 78.05]

**Mean Accuracy (Keras):** 68.94%

**Std Accuracy (Keras):** 9.05%

Interpretación. El promedio es comparable al obtenido con la implementación “from scratch” **(≈66%)**, lo cual corrobora que el modelo tiene un desempeño consistente en ese rango. La mayor variabilidad entre folds en Keras puede deberse a diferencias en la inicialización de pesos, el orden de los datos y el esquema de optimización por mini-batches.